In [ ]:
import igraph
from fa2_modified import ForceAtlas2
import random
import os
import pickle as pk

In [ ]:
def describe(g):
    print( g.vcount() )
    print( g.ecount() )
    print( g.vertex_attributes() )
    print( g.edge_attributes() )

In [ ]:
print("Available network graph files:")
for g in set( [g.split('_')[0] for g in os.listdir("graphs/")] ):
    print("-", g)

In [ ]:
dataset = 'LitLangMed'
explode = True

In [ ]:
if explode:
    gc = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_cocitation_graph_exploded.pk' )
else:
    gc = igraph.Graph.Read_Pickle( 'graphs/' + dataset + '_cocitation_graph.pk' )

describe(gc)

### Filter

In [ ]:
if explode:
    filt_thresh = 2
else:
    filt_thresh = 5

In [ ]:
# Delete edges with cocitation weight less than threshold
gc.es.select(weight_lt = filt_thresh).delete()

# Delete vertices with zero degree
gc.vs.select(_degree_lt=1).delete()

print( gc.vcount() )
print( gc.ecount() )

### Prepare for layout

In [ ]:
from igraph.drawing.colors import color_to_html_format
from IPython.display import HTML, display

In [ ]:
color_by = 'community' # community / continent / location / function / quality

In [ ]:
# Create color palette for communities
unique_communities = sorted( list( set(gc.vs[color_by]) ) )
palette = igraph.ClusterColoringPalette( n=len( unique_communities ) )

if color_by == 'continent':
    palette = igraph.PrecalculatedPalette(
        ["red", "blue", "white"]
    )

color_map = {
    cat: color_to_html_format(palette[i]) for i, cat in enumerate(unique_communities)
}

In [ ]:
# Color vertices
for v in gc.vs:   
    comm_i = unique_communities.index( v[color_by] )
    v['color'] = palette[comm_i]

# Store vertex strengths
vertex_strengths = gc.strength( weights=gc.es['weight'] )
for i, v in enumerate( gc.vs ):
    v['strength'] = vertex_strengths[i]
del vertex_strengths

### Run ForceAtlas2

In [ ]:
random.seed(1327)

forceatlas2 = ForceAtlas2(
                        # Behavior alternatives
                        outboundAttractionDistribution=False,  # Dissuade hubs
                        linLogMode=False,  # NOT IMPLEMENTED
                        adjustSizes=False,  # Prevent overlap (NOT IMPLEMENTED)
                        edgeWeightInfluence=1,

                        # Performance
                        jitterTolerance=1.0,  # Tolerance
                        barnesHutOptimize=True,
                        barnesHutTheta=1.2,
                        multiThreaded=False,  # NOT IMPLEMENTED

                        # Tuning
                        scalingRatio=1.0,
                        strongGravityMode=True,
                        gravity=1,

                        # Log
                        verbose=True)

l = forceatlas2.forceatlas2_igraph_layout( gc, pos=None, iterations=1000, weight_attr='weight' )

In [ ]:
# Write layout
if explode:
    with open("layouts/" + dataset + "_" + str(filt_thresh) +  "_exploded.pkl", "wb") as f:
        pk.dump(l, f)
else:
    with open("layouts/" + dataset + "_" + str(filt_thresh) +  ".pkl", "wb") as f:
        pk.dump(l, f)

## Plot

In [ ]:
if explode:
    with open("layouts/" + dataset + "_" + str(filt_thresh) + "_exploded.pkl", "rb") as f:
        l = pk.load(f)
else:
    with open("layouts/" + dataset + "_" + str(filt_thresh) + ".pkl", "rb") as f:
        l = pk.load(f)

In [ ]:
if explode:
    out_f = 'plots/' + dataset + "_" + str(filt_thresh) + '_' + color_by + '_cocitations_exploded.pdf'
else:
    out_f = 'plots/' + dataset + "_" + str(filt_thresh) + '_' + color_by + '_cocitations.pdf'

igraph.plot(
    gc,
    out_f,
    layout=l, 
    vertex_size = [(s/200)+3 for s in gc.vs['strength']],
    vertex_color = gc.vs['color'],
    #vertex_label = [v['name'] if v['strength'] >= 2000 else '' for v in gc.vs],
    #vertex_label = [v['community'] for v in gc.vs],
    #vertex_label_color = 'black',
    #vertex_label_size = [(s/250) for s in gc.vs['strength']],
    edge_width=0.15,
    bbox=(1000,1000),
    margin=20
)

In [ ]:
for comm, col in color_map.items():
    display(HTML(f'<div style="width: 20px; height: 20px; background-color: {col}; border: 1px solid #000;"></div>' + str(comm)))
    print("")

## Write coordinates and save for interactive visualization 

In [ ]:
gc.vs["x"], gc.vs["y"] = zip(*l.coords)

In [ ]:
if explode:
    gc.write_graphml("graphs/" + dataset + "_cocitation_communities_explode.graphml")
else:
    gc.write_graphml("graphs/" + dataset + "_cocitation_communities_explode.graphml")

# Get top authors in each community

In [ ]:
import pandas as pd
from collections import defaultdict

In [ ]:
n_authors = 20

In [ ]:
authors_by_community = defaultdict(list)

for v in gc.vs:
    authors_by_community[v['community']].append((v['authors'], v['strength']))

In [ ]:
top_authors_by_community = {}

for comm_id, authors in sorted( authors_by_community.items() ):
    
    authors_sorted = sorted(authors, key=lambda x: x[1], reverse=True)
    authors_sorted = [ au[0] + ', ' + str(au[1]) for au in authors_sorted]
    top_authors_by_community[comm_id] = authors_sorted[:n_authors]

In [ ]:
top_authors_df = pd.DataFrame.from_dict(top_authors_by_community, orient='index').T
top_authors_df = top_authors_df.add_prefix('Community ')
top_authors_df

In [ ]:
if explode:
    top_authors_df.to_excel('xlsx/' + dataset + '_top_authors_by_community_exploded.xlsx', index=False)
else:
    top_authors_df.to_excel('xlsx/' + dataset + '_top_authors_by_community.xlsx', index=False)